In [1]:
# 1) Install dependencies
!pip -q install trafilatura beautifulsoup4 lxml sentence-transformers transformers torch scikit-learn nltk keybert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 12.7 MB/s eta 0:00:00


In [9]:
# 2) Imports

import re
import math
import json
import numpy as np
import pandas as pd
import requests
import nltk
import trafilatura

from bs4 import BeautifulSoup
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from keybert import KeyBERT

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# 3) URLs

url1 = "https://climate.nasa.gov/effects/"
url2 = "https://www.un.org/en/climatechange/science/causes-effects-climate-change"

# -------------------------
# 4) Helper: clean text
# -------------------------
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\[[^\]]*\]', '', text)
    text = re.sub(r'\s([?.!,;:](?:\s|$))', r'\1', text)
    return text.strip()

In [5]:
# 4) Helper: clean text

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\[[^\]]*\]', '', text)
    text = re.sub(r'\s([?.!,;:](?:\s|$))', r'\1', text)
    return text.strip()

In [6]:
# 5) Helper: fetch article text

def fetch_article_text(url):
    """
    Try trafilatura first, then BeautifulSoup fallback.
    """
    try:
        downloaded = trafilatura.fetch_url(url)
        extracted = trafilatura.extract(downloaded, include_comments=False, include_tables=False)
        if extracted and len(extracted) > 500:
            return clean_text(extracted)
    except:
        pass

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Remove noisy elements
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside", "form"]):
        tag.decompose()

    text = soup.get_text(separator=" ")
    text = clean_text(text)
    return text

In [7]:
# 6) Get article texts

text1 = fetch_article_text(url1)
text2 = fetch_article_text(url2)

print("Article 1 length:", len(text1))
print("Article 2 length:", len(text2))
print("\nPreview Article 1:\n", text1[:700])
print("\nPreview Article 2:\n", text2[:700])

Article 1 length: 9473
Article 2 length: 8400

Preview Article 1:
 Takeaways - We already see effects scientists predicted, such as the loss of sea ice, melting glaciers and ice sheets, sea level rise, and more intense heat waves. - Scientists predict global temperature increases from human-made greenhouse gases will continue. Severe weather damage will also increase and intensify. Earth Will Continue to Warm and the Effects Will Be Profound Global climate change is not a future problem. Changes to Earth’s climate driven by increased human emissions of heat-trapping greenhouse gases are already having widespread effects on the environment: glaciers and ice sheets are shrinking, river and lake ice is breaking up earlier, plant and animal geographic ranges ar

Preview Article 2:
 Fossil fuels – coal, oil and gas – are by far the largest contributor to global climate change, accounting for around 68 per cent of global greenhouse gas emissions and nearly 90 per cent of all carbon dioxide e

In [10]:
# 7) Sentence splitting

sentences1 = sent_tokenize(text1)
sentences2 = sent_tokenize(text2)

print("\nNumber of sentences in Article 1:", len(sentences1))
print("Number of sentences in Article 2:", len(sentences2))


Number of sentences in Article 1: 80
Number of sentences in Article 2: 81


In [11]:
# 8) Similarity using Sentence Embeddings

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Document embeddings
doc_embedding1 = embedding_model.encode([text1], convert_to_numpy=True)
doc_embedding2 = embedding_model.encode([text2], convert_to_numpy=True)

doc_similarity = cosine_similarity(doc_embedding1, doc_embedding2)[0][0]

print("\n=== 1) ARTICLE SIMILARITY ===")
print(f"Cosine similarity between the two articles: {doc_similarity:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


=== 1) ARTICLE SIMILARITY ===
Cosine similarity between the two articles: 0.5069


In [14]:
# 9) Topic similarity using keywords + embeddings

kw_model = KeyBERT(model=embedding_model)

keywords1 = kw_model.extract_keywords(
    text1,
    keyphrase_ngram_range=(1, 2),
    stop_words='english',
    top_n=5
)

keywords2 = kw_model.extract_keywords(
    text2,
    keyphrase_ngram_range=(1, 2),
    stop_words='english',
    top_n=5
)

top_keywords1 = [kw for kw, score in keywords1]
top_keywords2 = [kw for kw, score in keywords2]

print("\n=== 3) TOP FIVE KEYWORDS ===")
print("Article 1 Keywords:", top_keywords1)
print("Article 2 Keywords:", top_keywords2)

# Topic overlap
common_keywords = set(top_keywords1).intersection(set(top_keywords2))
print("\nCommon keywords/topics:", common_keywords if common_keywords else "No exact keyword overlap")


=== 3) TOP FIVE KEYWORDS ===
Article 1 Keywords: ['climate change', 'climate changes', 'warming global', 'effects climate', 'warming climate']
Article 2 Keywords: ['global emissions', 'generates emissions', 'emissions worldwide', 'produce emissions', 'emissions burning']

Common keywords/topics: No exact keyword overlap


In [15]:
# 10) Chunking helper for long texts

def chunk_text_by_sentences(text, max_words=180):
    sents = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_len = 0

    for sent in sents:
        sent_len = len(sent.split())
        if current_len + sent_len <= max_words:
            current_chunk.append(sent)
            current_len += sent_len
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sent]
            current_len = sent_len

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

chunks1 = chunk_text_by_sentences(text1, max_words=180)
chunks2 = chunk_text_by_sentences(text2, max_words=180)

In [16]:
# 11) Sentiment analysis using transformer model

sentiment_model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
sentiment_pipe = pipeline(
    "text-classification",
    model=sentiment_model_name,
    tokenizer=sentiment_model_name,
    truncation=True
)

def aggregate_classifier_outputs(chunks, classifier):
    """
    Runs classifier on each chunk and returns:
    - average score per label
    - dominant label
    """
    label_scores = {}

    for chunk in chunks:
        result = classifier(chunk)[0]
        label = result["label"]
        score = result["score"]

        if label not in label_scores:
            label_scores[label] = []
        label_scores[label].append(score)

    avg_scores = {label: np.mean(scores) for label, scores in label_scores.items()}
    dominant_label = max(avg_scores, key=avg_scores.get)
    return avg_scores, dominant_label

sent_scores1, sent_label1 = aggregate_classifier_outputs(chunks1, sentiment_pipe)
sent_scores2, sent_label2 = aggregate_classifier_outputs(chunks2, sentiment_pipe)

print("\n=== 2b) SENTIMENT ===")
print("Article 1 Sentiment Scores:", sent_scores1)
print("Article 1 Dominant Sentiment:", sent_label1)
print("Article 2 Sentiment Scores:", sent_scores2)
print("Article 2 Dominant Sentiment:", sent_label2)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


=== 2b) SENTIMENT ===
Article 1 Sentiment Scores: {'neutral': np.float64(0.7378697295983633), 'negative': np.float64(0.5532447695732117), 'positive': np.float64(0.5358126759529114)}
Article 1 Dominant Sentiment: neutral
Article 2 Sentiment Scores: {'negative': np.float64(0.6691375573476156), 'neutral': np.float64(0.6514435708522797)}
Article 2 Dominant Sentiment: negative


In [18]:
# 12) Emotion analysis using transformer model

emotion_model_name = "j-hartmann/emotion-english-distilroberta-base"
emotion_pipe = pipeline(
    "text-classification",
    model=emotion_model_name,
    tokenizer=emotion_model_name,
    truncation=True
)

def aggregate_emotions(chunks, classifier):
    emotion_scores = {}

    for chunk in chunks:
        result = classifier(chunk, top_k=None)

        if len(result) > 0 and isinstance(result[0], list):
         result = result[0]

        for item in result:
            label = item["label"]
            score = item["score"]
            if label not in emotion_scores:
                emotion_scores[label] = []
            emotion_scores[label].append(score)

    avg_scores = {label: float(np.mean(scores)) for label, scores in emotion_scores.items()}
    sorted_emotions = dict(sorted(avg_scores.items(), key=lambda x: x[1], reverse=True))
    top_emotion = max(avg_scores, key=avg_scores.get)
    return sorted_emotions, top_emotion

emotion_scores1, top_emotion1 = aggregate_emotions(chunks1, emotion_pipe)
emotion_scores2, top_emotion2 = aggregate_emotions(chunks2, emotion_pipe)

print("\n=== 2c) EMOTIONS ===")
print("Article 1 Emotion Scores:", emotion_scores1)
print("Article 1 Top Emotion:", top_emotion1)
print("Article 2 Emotion Scores:", emotion_scores2)
print("Article 2 Top Emotion:", top_emotion2)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== 2c) EMOTIONS ===
Article 1 Emotion Scores: {'neutral': 0.6859559913476309, 'fear': 0.10872030775580141, 'anger': 0.07095786277204752, 'sadness': 0.052512749201721616, 'disgust': 0.045669670630660325, 'surprise': 0.028363707475364208, 'joy': 0.00781970222791036}
Article 1 Top Emotion: neutral
Article 2 Emotion Scores: {'neutral': 0.3784692892804742, 'anger': 0.1812184234149754, 'disgust': 0.17932322761043906, 'fear': 0.16460101818665862, 'sadness': 0.08394808624871075, 'surprise': 0.008807654434349388, 'joy': 0.003632323641795665}
Article 2 Top Emotion: neutral


In [19]:
# 13) Topic similarity check

topic_similarity_embedding = embedding_model.encode(
    [" ".join(top_keywords1), " ".join(top_keywords2)],
    convert_to_numpy=True
)

keyword_similarity = cosine_similarity(
    [topic_similarity_embedding[0]],
    [topic_similarity_embedding[1]]
)[0][0]

print("\n=== 2a) SIMILAR TOPICS? ===")
print(f"Keyword/topic similarity score: {keyword_similarity:.4f}")

if keyword_similarity >= 0.70:
    topic_conclusion = "Yes, both articles discuss highly similar topics."
elif keyword_similarity >= 0.45:
    topic_conclusion = "Yes, both articles discuss moderately similar topics."
else:
    topic_conclusion = "The articles have limited topic overlap."

print(topic_conclusion)


=== 2a) SIMILAR TOPICS? ===
Keyword/topic similarity score: 0.5128
Yes, both articles discuss moderately similar topics.


In [20]:
# 14) Prepare results dictionary

results = {
    "article_similarity_cosine": round(float(doc_similarity), 4),
    "topic_similarity_cosine": round(float(keyword_similarity), 4),
    "article_1": {
        "url": url1,
        "top_keywords": top_keywords1,
        "dominant_sentiment": sent_label1,
        "sentiment_scores": {k: round(float(v), 4) for k, v in sent_scores1.items()},
        "top_emotion": top_emotion1,
        "emotion_scores": {k: round(float(v), 4) for k, v in emotion_scores1.items()}
    },
    "article_2": {
        "url": url2,
        "top_keywords": top_keywords2,
        "dominant_sentiment": sent_label2,
        "sentiment_scores": {k: round(float(v), 4) for k, v in sent_scores2.items()},
        "top_emotion": top_emotion2,
        "emotion_scores": {k: round(float(v), 4) for k, v in emotion_scores2.items()}
    },
    "common_keywords": list(common_keywords),
    "topic_conclusion": topic_conclusion
}

print("\n=== RESULT SUMMARY (RAW) ===")
print(json.dumps(results, indent=2))


=== RESULT SUMMARY (RAW) ===
{
  "article_similarity_cosine": 0.5069,
  "topic_similarity_cosine": 0.5128,
  "article_1": {
    "url": "https://climate.nasa.gov/effects/",
    "top_keywords": [
      "climate change",
      "climate changes",
      "warming global",
      "effects climate",
      "warming climate"
    ],
    "dominant_sentiment": "neutral",
    "sentiment_scores": {
      "neutral": 0.7379,
      "negative": 0.5532,
      "positive": 0.5358
    },
    "top_emotion": "neutral",
    "emotion_scores": {
      "neutral": 0.686,
      "fear": 0.1087,
      "anger": 0.071,
      "sadness": 0.0525,
      "disgust": 0.0457,
      "surprise": 0.0284,
      "joy": 0.0078
    }
  },
  "article_2": {
    "url": "https://www.un.org/en/climatechange/science/causes-effects-climate-change",
    "top_keywords": [
      "global emissions",
      "generates emissions",
      "emissions worldwide",
      "produce emissions",
      "emissions burning"
    ],
    "dominant_sentiment": "neg

In [25]:
# 15) Summarize findings using an LLM

from transformers import pipeline

summary_prompt = f"""
You are helping with a homework assignment.

Summarize the comparison of two climate change articles in one clear academic paragraph.

Results:
- Article similarity cosine score: {results['article_similarity_cosine']}
- Topic similarity cosine score: {results['topic_similarity_cosine']}
- Article 1 keywords: {results['article_1']['top_keywords']}
- Article 2 keywords: {results['article_2']['top_keywords']}
- Common keywords: {results['common_keywords']}
- Article 1 sentiment: {results['article_1']['dominant_sentiment']}
- Article 2 sentiment: {results['article_2']['dominant_sentiment']}
- Article 1 top emotion: {results['article_1']['top_emotion']}
- Article 2 top emotion: {results['article_2']['top_emotion']}
- Topic conclusion: {results['topic_conclusion']}
"""

# Use text-generation instead of text2text-generation
llm_pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    tokenizer="google/flan-t5-base"
)

llm_output = llm_pipe(
    summary_prompt,
    max_new_tokens=180,
    do_sample=False
)

print("\n=== 4)LLM SUMMARY OF FINDINGS ===")
print(llm_output[0]["generated_text"])

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl


=== 4)LLM SUMMARY OF FINDINGS ===

You are helping with a homework assignment.

Summarize the comparison of two climate change articles in one clear academic paragraph.

Results:
- Article similarity cosine score: 0.5069
- Topic similarity cosine score: 0.5128
- Article 1 keywords: ['climate change', 'climate changes', 'warming global', 'effects climate', 'warming climate']
- Article 2 keywords: ['global emissions', 'generates emissions', 'emissions worldwide', 'produce emissions', 'emissions burning']
- Common keywords: []
- Article 1 sentiment: neutral
- Article 2 sentiment: negative
- Article 1 top emotion: neutral
- Article 2 top emotion: neutral
- Topic conclusion: Yes, both articles discuss moderately similar topics.



In [24]:
# neat final table

final_df = pd.DataFrame({
    "Metric": [
        "Document Similarity",
        "Topic Similarity",
        "Article 1 Sentiment",
        "Article 2 Sentiment",
        "Article 1 Top Emotion",
        "Article 2 Top Emotion",
        "Article 1 Keywords",
        "Article 2 Keywords"
    ],
    "Value": [
        round(float(doc_similarity), 4),
        round(float(keyword_similarity), 4),
        sent_label1,
        sent_label2,
        top_emotion1,
        top_emotion2,
        ", ".join(top_keywords1),
        ", ".join(top_keywords2)
    ]
})

print("\n FINAL RESULTS TABLE ")
display(final_df)


 FINAL RESULTS TABLE 


,Metric,Value
0,Document Similarity,0.5069
1,Topic Similarity,0.5128
2,Article 1 Sentiment,neutral
3,Article 2 Sentiment,negative
4,Article 1 Top Emotion,neutral
5,Article 2 Top Emotion,neutral
6,Article 1 Keywords,"climate change, climate changes, warming globa..."
7,Article 2 Keywords,"global emissions, generates emissions, emissio..."
